### Подключение

In [1]:
from system_check import *
CUDA_state_print()

CUDA доступна: True
Количество GPU: 4

GPU #0:
Название: Tesla V100-SXM3-32GB
Вычислительная способность (CUDA Capability): (7, 0)
Объем памяти: 31.73 GB

GPU #1:
Название: Tesla V100-SXM3-32GB
Вычислительная способность (CUDA Capability): (7, 0)
Объем памяти: 31.73 GB

GPU #2:
Название: Tesla V100-SXM3-32GB
Вычислительная способность (CUDA Capability): (7, 0)
Объем памяти: 31.73 GB

GPU #3:
Название: Tesla V100-SXM3-32GB
Вычислительная способность (CUDA Capability): (7, 0)
Объем памяти: 31.73 GB


In [2]:
import sys
sys.path.append("/global_functions")
from global_functions.global_functions import *

------------------------------------
---> Choosing resource
---> Result of torch.cuda.is_available():
True
---> Final resource
-----> CUDA: 1
-----> device: cuda:1
------------------------------------


/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
loading configuration file /home/pmartynyuk/UnTIE project/scripts/models_processing/models/eng_sentence_transformer_model/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_t

### Датасет

In [3]:
name = "scirex_structured.json"
datasets_path = "../datasets"

df_loaded = load_dataframe_from_json(f"{datasets_path}/{name}")

In [4]:
df_loaded = replace_underscores_in_array_column(df=df_loaded,
                                        column_name="tasks")
df_loaded.head()

,doc_id,methods,tasks,metrics,materials,score,original_text,tasks_cleaned
0,000f90380d768a85e2316225854fc377c079b5c4,[FRRN],"[Semantic_Segmentation, Real-Time_Semantic_Seg...","[Frame__fps_, Mean_IoU, mIoU, Time__ms_]",[Cityscapes],"[2.1, 71.8%, 469]",Full - Resolution Residual Networks for Semant...,"[Semantic Segmentation, Real-Time Semantic Seg..."
1,0012de6bec1f25599e4f02517637e531a71909b9,[V-Net___Dice-based_loss],[Volumetric_Medical_Image_Segmentation],[Dice_Score],[PROMISE_2012],[0.869],document: V - Net: Fully Convolutional Neural ...,[Volumetric Medical Image Segmentation]
2,007ab5528b3bd310a80d553cccad4b78dc496b02,"[BiDAF__single_model_, BiDAF, BiDAF__ensemble_...","[Question_Answering, Open-Domain_Question_Answ...","[METEOR, EM__Quasar-T_, F1__Quasar-T_, F1, EM,...","[NarrativeQA, MS_MARCO, SQuAD1_1, CNN___Daily_...","[33.45, 23.96, 15.69, 25.9, 79.6, 36.74, 76.9,...",document: Bi - Directional Attention Flow for ...,"[Question Answering, Open-Domain Question Answ..."
3,0095c269e7d0c990249312687fc43521019809c4,[50D_stacked_TC-LSTMs],[Natural_Language_Inference],"[Parameters, __Train_Accuracy, __Test_Accuracy]",[SNLI],"[190k, 86.7, 85.1]",document: Modelling Interaction of Sentence Pa...,[Natural Language Inference]
4,00b1cdc5bd77bf27f9b1ca630365eeeb456913b4,"[AlphaZero, AlphaGo_Zero]","[Game_of_Go, Game_of_Shogi]",[ELO_Rating],[ELO_Ratings],"[4650, 5185]",document: Mastering Chess and Shogi by Self - ...,"[Game of Go, Game of Shogi]"


### Модель

In [5]:
model_name = "scart_init_model.json"
model_params_path = "../model_params"

df_model_params = pd.read_json(f"{model_params_path}/{model_name}")

df_model_params

,model_name,description,fields
0,ScArt Model,Модель для обработки научных статей на английс...,"{'field_id': 1, 'field_name': 'task', 'field_t..."


In [6]:
field_name = df_model_params["fields"][0]["field_name"]
questions = df_model_params["fields"][0]["questions"]
keywords = df_model_params["fields"][0]["keywords"]
text = df_model_params["fields"][0]["text"]

In [7]:
them_aspect = ThematicAspect(field_name, convert_into_question_class(questions))

### Сэмпл

In [8]:
num_of_sample = 0

In [9]:
text_of_doc = df_loaded["original_text"][num_of_sample]

In [10]:
reference_answers = df_loaded["tasks_cleaned"][num_of_sample]
reference_answers

['Semantic Segmentation', 'Real-Time Semantic Segmentation']

### Обработка

In [11]:
text_proc = TextProcesser()
sents = text_proc.split_into_sentences(text_of_doc)

sentences = [Sentence(sents[i], i) for i in range(len(sents))]

print("---> Number of sentences:")
print(f"--> {len(sentences)}")

chunk_builder = ChunkBuilder()
text_chunks = chunk_builder.build_chunks(sentences)

print("---\n---> Number of chunks:")
print(f"--> {len(text_chunks)}")

---> Number of sentences:
--> 276
---
---> Number of chunks:
--> 22


#### Базовый ответ

In [12]:
simp_ans_finder = SimpleAnswerFinder()


question_with_answers = simp_ans_finder.find_answers(
        question = convert_into_question_class(questions)[0],
        chunks = text_chunks
        )

Device set to use cuda:1
Disabling tokenizer parallelism, we're using DataLoader multithreading already
RobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [13]:
print("\n---\n---> Founded answers:")
for ans in question_with_answers.answers:
    print(ans.text)
    print(ans.confidence)
    print(ans.chunk.text)
    print("----------------------\n")


---
---> Founded answers:
overfitting
0.04313625767827034
We will make code for the gradient computation for arbitrary networks publicly available in Theano / Lasagne. In order to reduce overfitting, we used two methods of data augmentation: translation augmentation and gamma augmentation. The former method randomly translates an image and its annotations. In order to keep consistent image dimensions, we have to pad the translated images and annotations. To this end, we use reflection padding on the image and constant padding with void labels on the annotations. Our second method of data augmentation is gamma augmentation. We use a slightly modified gamma augmentation method detailed in Appendix A. section: Experimental Evaluation We evaluate our approach on the recently released Cityscapes benchmark [reference] containing images recorded in 50 different cities. This benchmark provides 5, 000 images with high - quality annotations split up into a training, validation, and test set (2,

In [14]:
filtering_set = FilteringSet(text, keywords)

res1 = extract_short_answer(text=text_of_doc,
                them_asp=them_aspect, 
                filter_mode=FilterMode.NO_FILTER,
                filter_set=filtering_set,
                agg_mode=AggMode.NO_AGG)

Device set to use cuda:1


In [15]:
print(res1['final_answer'].text)

for ans in res1['final_answer'].supporting_answers:
    print(ans.text)
    print(ans.chunk.text)


semantic segmentation
semantic segmentation
1 https: // github.com / TobyPDE / FRRN section: Related Work The dramatic performance improvements from using CNNs for semantic segmentation have brought about an increasing demand for such algorithms in the context of autonomous driving scenarios. As a large amount of annotated data is crucial in order to train such deep networks, multiple new datasets have been released to encourage further research in this area, including Synthia [reference], Virtual KITTI [reference], and Cityscapes [reference]. In this work, we focus on Cityscapes, a recent large - scale dataset consisting of real - world imagery with well - curated annotations. Given their success, we will constrain our literature review to deep learning based semantic segmentation approaches and deep learning network architectures. Semantic Segmentation Approaches. Over the last years, the most successful semantic segmentation approaches have been based on convolutional neural network

In [16]:
validator = AnswerValidator()
validator_result = validator.validate_answers(question_with_answers.answers, reference_answers[0])

In [17]:
validator_result

[Answer(text='semantic segmentation', chunk=TextChunk(sentences=[Sentence(text='1 https: // github.com / TobyPDE / FRRN section: Related Work The dramatic performance improvements from using CNNs for semantic segmentation have brought about an increasing demand for such algorithms in the context of autonomous driving scenarios.', num=45), Sentence(text='As a large amount of annotated data is crucial in order to train such deep networks, multiple new datasets have been released to encourage further research in this area, including Synthia [reference], Virtual KITTI [reference], and Cityscapes [reference].', num=46), Sentence(text='In this work, we focus on Cityscapes, a recent large - scale dataset consisting of real - world imagery with well - curated annotations.', num=47), Sentence(text='Given their success, we will constrain our literature review to deep learning based semantic segmentation approaches and deep learning network architectures.', num=48), Sentence(text='Semantic Segmen

In [18]:
valid_chunks = [ans.chunk.text for ans in validator_result]

print("Искомые фрагменты:")
for chunk in valid_chunks:
    print("\t" + chunk)

Искомые фрагменты:
	1 https: // github.com / TobyPDE / FRRN section: Related Work The dramatic performance improvements from using CNNs for semantic segmentation have brought about an increasing demand for such algorithms in the context of autonomous driving scenarios. As a large amount of annotated data is crucial in order to train such deep networks, multiple new datasets have been released to encourage further research in this area, including Synthia [reference], Virtual KITTI [reference], and Cityscapes [reference]. In this work, we focus on Cityscapes, a recent large - scale dataset consisting of real - world imagery with well - curated annotations. Given their success, we will constrain our literature review to deep learning based semantic segmentation approaches and deep learning network architectures. Semantic Segmentation Approaches. Over the last years, the most successful semantic segmentation approaches have been based on convolutional neural networks (CNNs). Early approach

In [19]:
dicts_with_attention_keywords = []

for chunk in valid_chunks:
    key_dict = get_qa_attention_weights(model = BERTEngQASingleton(),
                                    question = questions[0], 
                                    context = chunk,
                                    device = get_device())
    dicts_with_attention_keywords.append(key_dict)

dicts_with_attention_keywords

[{'tokens': ['</s>',
   '1',
   'Ġhttps',
   ':',
   'Ġ//',
   'Ġgithub',
   '.',
   'com',
   'Ġ/',
   'ĠToby',
   'PD',
   'E',
   'Ġ/',
   'ĠFR',
   'RN',
   'Ġsection',
   ':',
   'ĠRelated',
   'ĠWork',
   'ĠThe',
   'Ġdramatic',
   'Ġperformance',
   'Ġimprovements',
   'Ġfrom',
   'Ġusing',
   'ĠCNN',
   's',
   'Ġfor',
   'Ġsemantic',
   'Ġsegment',
   'ation',
   'Ġhave',
   'Ġbrought',
   'Ġabout',
   'Ġan',
   'Ġincreasing',
   'Ġdemand',
   'Ġfor',
   'Ġsuch',
   'Ġalgorithms',
   'Ġin',
   'Ġthe',
   'Ġcontext',
   'Ġof',
   'Ġautonomous',
   'Ġdriving',
   'Ġscenarios',
   '.',
   'ĠAs',
   'Ġa',
   'Ġlarge',
   'Ġamount',
   'Ġof',
   'Ġannot',
   'ated',
   'Ġdata',
   'Ġis',
   'Ġcrucial',
   'Ġin',
   'Ġorder',
   'Ġto',
   'Ġtrain',
   'Ġsuch',
   'Ġdeep',
   'Ġnetworks',
   ',',
   'Ġmultiple',
   'Ġnew',
   'Ġdatasets',
   'Ġhave',
   'Ġbeen',
   'Ġreleased',
   'Ġto',
   'Ġencourage',
   'Ġfurther',
   'Ġresearch',
   'Ġin',
   'Ġthis',
   'Ġarea',
   ',',
   'Ġin

In [20]:
len(dicts_with_attention_keywords)

2

In [21]:
filtered_valid_chunks = dymamic_filter_uniform_words(valid_chunks, IDF_THRESHOLD)
filtered_valid_chunks

(['https frrn section relate dramatic improvement cnn bring about increase demand algorithm context autonomous drive annotate crucial order deep multiple new dataset release encourage far research area include virtual cityscape focus cityscape recent scale dataset consist real world imagery well curate annotation give their success will constrain literature review deep learn base deep learn over last year most base neural early constrain their bottom up follow base rather than classify entire first place et al perform wise feature originate multiple scale follow aggregation these noisy prediction over superpixel introduction so call long et al open wide range research end end long et al far reformulate popular enable pretraine skip connection add which allow information propagate directly early layer deeply layer',
  'reference reference reference reference task assign set predefine class label important tool complex relationship entity usually find street scene car pedestrian road or 

In [22]:
filtered_valid_chunks[0]

['https frrn section relate dramatic improvement cnn bring about increase demand algorithm context autonomous drive annotate crucial order deep multiple new dataset release encourage far research area include virtual cityscape focus cityscape recent scale dataset consist real world imagery well curate annotation give their success will constrain literature review deep learn base deep learn over last year most base neural early constrain their bottom up follow base rather than classify entire first place et al perform wise feature originate multiple scale follow aggregation these noisy prediction over superpixel introduction so call long et al open wide range research end end long et al far reformulate popular enable pretraine skip connection add which allow information propagate directly early layer deeply layer',
 'reference reference reference reference task assign set predefine class label important tool complex relationship entity usually find street scene car pedestrian road or si

In [23]:
cleaned_keywords_by_chunks = []

for key_dict in dicts_with_attention_keywords:

        cleaned_keys_batch = get_keywords_top_attention_clean(key_dict['tokens'], 
                key_dict['attention_weights'], top_k=100)

        cleaned_keywords_by_chunks.append(cleaned_keys_batch)

cleaned_keywords_by_chunks

[[{'word': 'the', 'weight': np.float32(0.004204311)},
  {'word': 'the', 'weight': np.float32(0.003936652)},
  {'word': 'performs', 'weight': np.float32(0.0039064568)},
  {'word': 'on', 'weight': np.float32(0.003398265)},
  {'word': 'training', 'weight': np.float32(0.003206912)},
  {'word': 'constrained', 'weight': np.float32(0.0031933999)},
  {'word': 'opened', 'weight': np.float32(0.0031924634)},
  {'word': 'we', 'weight': np.float32(0.0031579488)},
  {'word': 'classification', 'weight': np.float32(0.0030524763)},
  {'word': 'research', 'weight': np.float32(0.0028580574)},
  {'word': 'classification', 'weight': np.float32(0.002821498)},
  {'word': 'early', 'weight': np.float32(0.0028180352)},
  {'word': 'train', 'weight': np.float32(0.0027581344)},
  {'word': 'to', 'weight': np.float32(0.0027053903)},
  {'word': 'the', 'weight': np.float32(0.0027015836)},
  {'word': 'dataset', 'weight': np.float32(0.0026851872)},
  {'word': 'work', 'weight': np.float32(0.002657178)},
  {'word': 'as', 

In [24]:
len(cleaned_keywords_by_chunks)

2

In [25]:
filtered_keywords_by_chunks = []

for key_dict, filtered_chunk in zip(cleaned_keywords_by_chunks, filtered_valid_chunks[0]):

    filtered_keywords_by_chunks.append(filter_words_by_text(key_dict, filtered_chunk))

filtered_keywords_by_chunks

[[{'word': 'research', 'weight': np.float32(0.0028580574)},
  {'word': 'early', 'weight': np.float32(0.0028180352)},
  {'word': 'dataset', 'weight': np.float32(0.0026851872)},
  {'word': 'rather', 'weight': np.float32(0.002575087)},
  {'word': 'neural', 'weight': np.float32(0.002452692)},
  {'word': 'introduction', 'weight': np.float32(0.0024521477)},
  {'word': 'crucial', 'weight': np.float32(0.002284828)},
  {'word': 'aggregation', 'weight': np.float32(0.0022236684)},
  {'word': 'research', 'weight': np.float32(0.0022175685)},
  {'word': 'will', 'weight': np.float32(0.0021956167)},
  {'word': 'imagery', 'weight': np.float32(0.0021955133)},
  {'word': 'success', 'weight': np.float32(0.0021803232)},
  {'word': 'cnn', 'weight': np.float32(0.0020930015)},
  {'word': 'long', 'weight': np.float32(0.0020724086)},
  {'word': 'autonomous', 'weight': np.float32(0.00206999)},
  {'word': 'over', 'weight': np.float32(0.00203886)},
  {'word': 'which', 'weight': np.float32(0.0019957756)},
  {'word'

In [26]:
all_filtered_keywords = merge_and_deduplicate_word_arrays(filtered_keywords_by_chunks)

all_filtered_keywords

[{'word': 'task', 'weight': 0.0036683217622339725},
 {'word': 'it', 'weight': 0.003639453323557973},
 {'word': 'current', 'weight': 0.003179467748850584},
 {'word': 'employ', 'weight': 0.0030397071968764067},
 {'word': 'research', 'weight': 0.0028580573853105307},
 {'word': 'example', 'weight': 0.0028540880884975195},
 {'word': 'discard', 'weight': 0.002842721063643694},
 {'word': 'all', 'weight': 0.0028261032421141863},
 {'word': 'early', 'weight': 0.0028180351946502924},
 {'word': 'that', 'weight': 0.002759756753221154},
 {'word': 'dataset', 'weight': 0.0026851871516555548},
 {'word': 'require', 'weight': 0.002683906350284815},
 {'word': 'rather', 'weight': 0.002575086895376444},
 {'word': 'automotive', 'weight': 0.002558482810854912},
 {'word': 'or', 'weight': 0.0025317275431007147},
 {'word': 'where', 'weight': 0.0024977289140224457},
 {'word': 'many', 'weight': 0.0024618273600935936},
 {'word': 'neural', 'weight': 0.0024526920169591904},
 {'word': 'introduction', 'weight': 0.00245

In [27]:
all_filtered_keywords = postprocess_words_array(all_filtered_keywords)
all_filtered_keywords

[{'word': 'task', 'weight': 0.0036683217622339725},
 {'word': 'current', 'weight': 0.003179467748850584},
 {'word': 'employ', 'weight': 0.0030397071968764067},
 {'word': 'research', 'weight': 0.0028580573853105307},
 {'word': 'example', 'weight': 0.0028540880884975195},
 {'word': 'discard', 'weight': 0.002842721063643694},
 {'word': 'dataset', 'weight': 0.0026851871516555548},
 {'word': 'require', 'weight': 0.002683906350284815},
 {'word': 'automotive', 'weight': 0.002558482810854912},
 {'word': 'neural', 'weight': 0.0024526920169591904},
 {'word': 'introduction', 'weight': 0.0024521476589143276},
 {'word': 'detection', 'weight': 0.0024286305997520685},
 {'word': 'auxiliary', 'weight': 0.0023717775475233793},
 {'word': 'crucial', 'weight': 0.00228482810780406},
 {'word': 'aggregation', 'weight': 0.002223668387159705},
 {'word': 'pursue', 'weight': 0.0022047029342502356},
 {'word': 'imagery', 'weight': 0.002195513341575861},
 {'word': 'success', 'weight': 0.002180323237553239},
 {'word'

In [28]:
ls_filtered_keywords = get_lemm_stemm_words(all_filtered_keywords)

In [29]:
ls_filtered_keywords

[{'word': 'task',
  'lemma': 'task',
  'stem': 'task',
  'weight': 0.0036683217622339725},
 {'word': 'current',
  'lemma': 'current',
  'stem': 'current',
  'weight': 0.003179467748850584},
 {'word': 'employ',
  'lemma': 'employ',
  'stem': 'employ',
  'weight': 0.0030397071968764067},
 {'word': 'research',
  'lemma': 'research',
  'stem': 'research',
  'weight': 0.0028580573853105307},
 {'word': 'example',
  'lemma': 'example',
  'stem': 'exampl',
  'weight': 0.0028540880884975195},
 {'word': 'discard',
  'lemma': 'discard',
  'stem': 'discard',
  'weight': 0.002842721063643694},
 {'word': 'dataset',
  'lemma': 'dataset',
  'stem': 'dataset',
  'weight': 0.0026851871516555548},
 {'word': 'require',
  'lemma': 'require',
  'stem': 'requir',
  'weight': 0.002683906350284815},
 {'word': 'automotive',
  'lemma': 'automotive',
  'stem': 'automot',
  'weight': 0.002558482810854912},
 {'word': 'neural',
  'lemma': 'neural',
  'stem': 'neural',
  'weight': 0.0024526920169591904},
 {'word': 'i

In [30]:
ls_filtered_keywords = filter_keywords_from_dict(
                            keywords_with_weights = ls_filtered_keywords,
                            them_asp = them_aspect,
                            antireference = reference_answers[0]
)

ls_filtered_keywords

[{'word': 'task',
  'lemma': 'task',
  'stem': 'task',
  'weight': 0.0036683217622339725,
  'score_diff': np.float32(0.40358943)},
 {'word': 'require',
  'lemma': 'require',
  'stem': 'requir',
  'weight': 0.002683906350284815,
  'score_diff': np.float32(0.30358487)},
 {'word': 'employ',
  'lemma': 'employ',
  'stem': 'employ',
  'weight': 0.0030397071968764067,
  'score_diff': np.float32(0.2867719)},
 {'word': 'crucial',
  'lemma': 'crucial',
  'stem': 'crucial',
  'weight': 0.00228482810780406,
  'score_diff': np.float32(0.26436627)},
 {'word': 'focus',
  'lemma': 'focus',
  'stem': 'focus',
  'weight': 0.001978835090994835,
  'score_diff': np.float32(0.21515632)},
 {'word': 'long',
  'lemma': 'long',
  'stem': 'long',
  'weight': 0.002072408562526107,
  'score_diff': np.float32(0.20282865)},
 {'word': 'application',
  'lemma': 'application',
  'stem': 'applic',
  'weight': 0.0020525797735899687,
  'score_diff': np.float32(0.17649734)},
 {'word': 'research',
  'lemma': 'research',
  

In [31]:
len(ls_filtered_keywords)

18

In [32]:
def extra_score_chunks_by_keywords(
    chunks: List[TextChunk], 
    keywords: List[dict],
    case_sensitive: bool = False,
    use_combined_weights: bool = True,
    weight_ratio: float = 0.5,  # Соотношение между weight и score_diff (0.5 = равное значение)
    min_matches: int = 1
) -> List[ScoredChunk]:
    """
    Оценивает фрагменты на основе количества и важности ключевых слов.
    Учитывает оба веса: weight (из модели внимания) и score_diff (из сравнения с эталонами).
    
    Параметры:
        chunks: Список объектов TextChunk для оценки
        keywords: Список словарей формата [{"word": str, "lemma": str, "stem": str, 
                 "weight": float, "score_diff": float}, ...]
        case_sensitive: Учитывать регистр при поиске
        use_combined_weights: Использовать комбинированные веса (weight и score_diff)
        weight_ratio: Соотношение между weight и score_diff (0.0 - только score_diff, 1.0 - только weight)
        min_matches: Минимальное количество совпадений для включения фрагмента в результат
    
    Возвращает:
        Список объектов ScoredChunk с оценками и информацией о совпадениях
    """
    if not keywords or not chunks:
        return []
    
    # Подготавливаем ключевые слова и их комбинированные веса
    keyword_weights = {}
    search_terms = set()
    term_to_keyword = {}
    
    for keyword in keywords:
        word = keyword['word']
        
        # Вычисляем комбинированный вес
        weight = keyword.get('weight', 1.0)
        score_diff = keyword.get('score_diff', 1.0)
        
        if use_combined_weights:
            # Комбинируем веса с учетом соотношения
            combined_weight = (weight * weight_ratio) + (score_diff * (1 - weight_ratio))
        else:
            # Используем только score_diff (по умолчанию)
            combined_weight = score_diff
        
        keyword_weights[word] = combined_weight
        
        # Добавляем лемму и стем для поиска
        lemma = keyword.get('lemma', word)
        stem = keyword.get('stem', word)
        
        search_terms.add(lemma)
        search_terms.add(stem)
        
        term_to_keyword[lemma.lower()] = word
        term_to_keyword[stem.lower()] = word
    
    # Создаем regex-паттерн для поиска
    pattern = re.compile(
        r'\b(?:{})\b'.format('|'.join(map(re.escape, search_terms))),
        flags=0 if case_sensitive else re.IGNORECASE
    )
    
    scored_chunks = []
    
    for chunk in chunks:
        # Находим все совпадения терминов в тексте
        text_to_search = chunk.text if case_sensitive else chunk.text.lower()
        found_terms = pattern.findall(text_to_search)
        
        # Собираем уникальные найденные ключевые слова и их веса
        matched_keywords = set()
        total_score = 0.0
        
        for term in found_terms:
            normalized_term = term if case_sensitive else term.lower()
            if normalized_term in term_to_keyword:
                keyword_word = term_to_keyword[normalized_term]
                if keyword_word not in matched_keywords:
                    matched_keywords.add(keyword_word)
                    total_score += keyword_weights.get(keyword_word, 1.0)
        
        # Применяем дополнительные факторы для улучшения оценки
        if matched_keywords:
            # Нормализуем по количеству слов в фрагменте
            normalization_factor = 1.0 + (len(matched_keywords) / max(1, chunk.word_token_count / 10))
            
            # Учитываем плотность ключевых слов
            density_bonus = min(2.0, 1.0 + (len(matched_keywords) / max(1, len(chunk.text.split()) / 20)))
            
            final_score = total_score * normalization_factor * density_bonus
            
            if len(matched_keywords) >= min_matches:
                scored_chunks.append(ScoredChunk(
                    chunk=chunk,
                    score=final_score,
                    matched_keywords=list(matched_keywords),
                    keyword_scores={kw: keyword_weights.get(kw, 1.0) for kw in matched_keywords},
                    # Добавляем информацию об исходных весах
                    original_weights={kw: {
                        'weight': next((k['weight'] for k in keywords if k['word'] == kw), 1.0),
                        'score_diff': next((k['score_diff'] for k in keywords if k['word'] == kw), 1.0)
                    } for kw in matched_keywords}
                ))
    
    # Сортируем по убыванию оценки
    scored_chunks.sort(key=lambda x: x.score, reverse=True)
    
    return scored_chunks

# Расширенная версия с учетом обоих весов
def extra_score_chunks_advanced(
    chunks: List[TextChunk], 
    keywords: List[dict],
    case_sensitive: bool = False,
    min_matches: int = 1,
    position_weight: float = 0.3,
    frequency_weight: float = 0.7,
    weight_ratio: float = 0.5  # Соотношение между weight и score_diff
) -> List[ScoredChunk]:
    """
    Расширенная версия оценки фрагментов с учетом позиции и частоты ключевых слов.
    Учитывает оба веса: weight и score_diff.
    """
    if not keywords or not chunks:
        return []
    
    # Подготавливаем ключевые слова с комбинированными весами
    keyword_weights = {}
    search_terms = set()
    term_to_keyword = {}
    
    for keyword in keywords:
        word = keyword['word']
        
        # Комбинируем веса
        weight = keyword.get('weight', 1.0)
        score_diff = keyword.get('score_diff', 1.0)
        combined_weight = (weight * weight_ratio) + (score_diff * (1 - weight_ratio))
        
        keyword_weights[word] = combined_weight
        
        lemma = keyword.get('lemma', word)
        stem = keyword.get('stem', word)
        
        search_terms.add(lemma)
        search_terms.add(stem)
        
        term_to_keyword[lemma.lower()] = word
        term_to_keyword[stem.lower()] = word
    
    pattern = re.compile(
        r'\b(?:{})\b'.format('|'.join(map(re.escape, search_terms))),
        flags=0 if case_sensitive else re.IGNORECASE
    )
    
    scored_chunks = []
    
    for chunk in chunks:
        text_to_search = chunk.text if case_sensitive else chunk.text.lower()
        found_matches = list(pattern.finditer(text_to_search))
        
        if not found_matches:
            continue
        
        # Анализируем совпадения
        matched_keywords = set()
        position_scores = []
        keyword_frequencies = {}
        
        for match in found_matches:
            term = match.group()
            normalized_term = term if case_sensitive else term.lower()
            
            if normalized_term in term_to_keyword:
                keyword_word = term_to_keyword[normalized_term]
                matched_keywords.add(keyword_word)
                
                # Оценка позиции (слова в начале текста более важны)
                position = match.start() / max(1, len(text_to_search))
                position_score = 1.0 - position  # чем раньше, тем выше оценка
                position_scores.append(position_score)
                
                # Подсчет частоты
                keyword_frequencies[keyword_word] = keyword_frequencies.get(keyword_word, 0) + 1
        
        if len(matched_keywords) >= min_matches:
            # Вычисляем общую оценку с учетом комбинированных весов
            base_score = sum(keyword_weights.get(kw, 1.0) for kw in matched_keywords)
            
            # Учитываем позицию (средняя оценка позиций)
            avg_position_score = np.mean(position_scores) if position_scores else 0.5
            position_contribution = 1.0 + position_weight * avg_position_score
            
            # Учитываем частоту (логарифмическая шкала для избежания перекоса)
            freq_contribution = 1.0 + frequency_weight * np.log1p(sum(keyword_frequencies.values()))
            
            # Учитываем уникальность (количество разных ключевых слов)
            uniqueness_bonus = 1.0 + (len(matched_keywords) / len(keywords)) * 0.5
            
            final_score = base_score * position_contribution * freq_contribution * uniqueness_bonus
            
            scored_chunks.append(ScoredChunk(
                chunk=chunk,
                score=final_score,
                matched_keywords=list(matched_keywords),
                keyword_scores=keyword_frequencies,
                # Добавляем информацию об исходных весах
                original_weights={kw: {
                    'weight': next((k['weight'] for k in keywords if k['word'] == kw), 1.0),
                    'score_diff': next((k['score_diff'] for k in keywords if k['word'] == kw), 1.0),
                    'combined_weight': keyword_weights.get(kw, 1.0)
                } for kw in matched_keywords}
            ))
    
    scored_chunks.sort(key=lambda x: x.score, reverse=True)
    return scored_chunks

In [33]:
# Использование только score_diff
scored_1 = extra_score_chunks_by_keywords(text_chunks, ls_filtered_keywords, 
                                        weight_ratio=0.0)

print("Топ-5 фрагментов по оценке:")
for i, scored_chunk in enumerate(scored_1[:5]):
    print(f"{i+1}. Оценка: {scored_chunk.score:.2f}")
    print(f"   Совпадения: {', '.join(scored_chunk.matched_keywords)}")
    print(f"   Веса: {scored_chunk.original_weights}")
    print(f"   Текст: {scored_chunk.chunk.text[:100]}...")
    print()

Топ-5 фрагментов по оценке:
1. Оценка: 4.26
   Совпадения: auxiliary, application, employ, current, target, example, detection, task, pursue, require, class
   Веса: {'auxiliary': {'weight': 0.0023717775475233793, 'score_diff': np.float32(0.16079235)}, 'application': {'weight': 0.0020525797735899687, 'score_diff': np.float32(0.17649734)}, 'employ': {'weight': 0.0030397071968764067, 'score_diff': np.float32(0.2867719)}, 'current': {'weight': 0.003179467748850584, 'score_diff': np.float32(0.04848516)}, 'target': {'weight': 0.0020016799680888653, 'score_diff': np.float32(0.15338182)}, 'example': {'weight': 0.0028540880884975195, 'score_diff': np.float32(0.041388273)}, 'detection': {'weight': 0.0024286305997520685, 'score_diff': np.float32(0.123521805)}, 'task': {'weight': 0.0036683217622339725, 'score_diff': np.float32(0.40358943)}, 'pursue': {'weight': 0.0022047029342502356, 'score_diff': np.float32(0.1143105)}, 'require': {'weight': 0.002683906350284815, 'score_diff': np.float32(0.30358

In [34]:
# Использование только weight
scored_2 = extra_score_chunks_by_keywords(text_chunks, ls_filtered_keywords, 
                                        weight_ratio=1.0)

print("Топ-5 фрагментов по оценке:")
for i, scored_chunk in enumerate(scored_2[:5]):
    print(f"{i+1}. Оценка: {scored_chunk.score:.2f}")
    print(f"   Совпадения: {', '.join(scored_chunk.matched_keywords)}")
    print(f"   Веса: {scored_chunk.original_weights}")
    print(f"   Текст: {scored_chunk.chunk.text[:100]}...")
    print()

Топ-5 фрагментов по оценке:
1. Оценка: 0.06
   Совпадения: auxiliary, application, employ, current, target, example, detection, task, pursue, require, class
   Веса: {'auxiliary': {'weight': 0.0023717775475233793, 'score_diff': np.float32(0.16079235)}, 'application': {'weight': 0.0020525797735899687, 'score_diff': np.float32(0.17649734)}, 'employ': {'weight': 0.0030397071968764067, 'score_diff': np.float32(0.2867719)}, 'current': {'weight': 0.003179467748850584, 'score_diff': np.float32(0.04848516)}, 'target': {'weight': 0.0020016799680888653, 'score_diff': np.float32(0.15338182)}, 'example': {'weight': 0.0028540880884975195, 'score_diff': np.float32(0.041388273)}, 'detection': {'weight': 0.0024286305997520685, 'score_diff': np.float32(0.123521805)}, 'task': {'weight': 0.0036683217622339725, 'score_diff': np.float32(0.40358943)}, 'pursue': {'weight': 0.0022047029342502356, 'score_diff': np.float32(0.1143105)}, 'require': {'weight': 0.002683906350284815, 'score_diff': np.float32(0.30358

In [35]:
# Равное сочетание обоих весов (по умолчанию)
scored_3 = extra_score_chunks_by_keywords(text_chunks, ls_filtered_keywords, 
                                        weight_ratio=0.5)

print("Топ-5 фрагментов по оценке:")
for i, scored_chunk in enumerate(scored_3[:5]):
    print(f"{i+1}. Оценка: {scored_chunk.score:.2f}")
    print(f"   Совпадения: {', '.join(scored_chunk.matched_keywords)}")
    print(f"   Веса: {scored_chunk.original_weights}")
    print(f"   Текст: {scored_chunk.chunk.text[:100]}...")
    print()

Топ-5 фрагментов по оценке:
1. Оценка: 2.16
   Совпадения: auxiliary, application, employ, current, target, example, detection, task, pursue, require, class
   Веса: {'auxiliary': {'weight': 0.0023717775475233793, 'score_diff': np.float32(0.16079235)}, 'application': {'weight': 0.0020525797735899687, 'score_diff': np.float32(0.17649734)}, 'employ': {'weight': 0.0030397071968764067, 'score_diff': np.float32(0.2867719)}, 'current': {'weight': 0.003179467748850584, 'score_diff': np.float32(0.04848516)}, 'target': {'weight': 0.0020016799680888653, 'score_diff': np.float32(0.15338182)}, 'example': {'weight': 0.0028540880884975195, 'score_diff': np.float32(0.041388273)}, 'detection': {'weight': 0.0024286305997520685, 'score_diff': np.float32(0.123521805)}, 'task': {'weight': 0.0036683217622339725, 'score_diff': np.float32(0.40358943)}, 'pursue': {'weight': 0.0022047029342502356, 'score_diff': np.float32(0.1143105)}, 'require': {'weight': 0.002683906350284815, 'score_diff': np.float32(0.30358

In [36]:
# Преимущество weight (70% weight, 30% score_diff)
scored_4 = extra_score_chunks_by_keywords(text_chunks, ls_filtered_keywords, 
                                        weight_ratio=0.7)

print("Топ-5 фрагментов по оценке:")
for i, scored_chunk in enumerate(scored_4[:5]):
    print(f"{i+1}. Оценка: {scored_chunk.score:.2f}")
    print(f"   Совпадения: {', '.join(scored_chunk.matched_keywords)}")
    print(f"   Веса: {scored_chunk.original_weights}")
    print(f"   Текст: {scored_chunk.chunk.text[:100]}...")
    print()

Топ-5 фрагментов по оценке:
1. Оценка: 1.32
   Совпадения: auxiliary, application, employ, current, target, example, detection, task, pursue, require, class
   Веса: {'auxiliary': {'weight': 0.0023717775475233793, 'score_diff': np.float32(0.16079235)}, 'application': {'weight': 0.0020525797735899687, 'score_diff': np.float32(0.17649734)}, 'employ': {'weight': 0.0030397071968764067, 'score_diff': np.float32(0.2867719)}, 'current': {'weight': 0.003179467748850584, 'score_diff': np.float32(0.04848516)}, 'target': {'weight': 0.0020016799680888653, 'score_diff': np.float32(0.15338182)}, 'example': {'weight': 0.0028540880884975195, 'score_diff': np.float32(0.041388273)}, 'detection': {'weight': 0.0024286305997520685, 'score_diff': np.float32(0.123521805)}, 'task': {'weight': 0.0036683217622339725, 'score_diff': np.float32(0.40358943)}, 'pursue': {'weight': 0.0022047029342502356, 'score_diff': np.float32(0.1143105)}, 'require': {'weight': 0.002683906350284815, 'score_diff': np.float32(0.30358

In [37]:
simp_ans_finder = ScoredAnswerFinder()


question_with_answers1 = simp_ans_finder.find_answers(
        question = convert_into_question_class(questions)[0],
        scored_chunks = scored_1
        )


print("\n---\n---> Founded answers:")
for ans in question_with_answers1.answers:
    print(ans.text)
    print(ans.chunk.score)
    print(ans.confidence)
    print(ans.chunk.chunk.text)
    print("----------------------\n")

Device set to use cuda:1



---
---> Founded answers:
Semantic image segmentation
4.2634473
3.913524415111169e-05
Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks. In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference]. Example output and the abstract structure of our fullresolution residual network. The network has two processing streams. The residual stream (blue) stays at the full image resolution, the pooling stream (red) undergoes a sequence of pooling and unpooling operations. The two processing streams are coupled using full - 

In [38]:
simp_ans_finder = ScoredAnswerFinder()


question_with_answers2 = simp_ans_finder.find_answers(
        question = convert_into_question_class(questions)[0],
        scored_chunks = scored_2
        )


print("\n---\n---> Founded answers:")
for ans in question_with_answers2.answers:
    print(ans.text)
    print(ans.chunk.score)
    print(ans.confidence)
    print(ans.chunk.chunk.text)
    print("----------------------\n")

Device set to use cuda:1



---
---> Founded answers:
Semantic image segmentation
0.06486309
3.913524415111169e-05
Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks. In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference]. Example output and the abstract structure of our fullresolution residual network. The network has two processing streams. The residual stream (blue) stays at the full image resolution, the pooling stream (red) undergoes a sequence of pooling and unpooling operations. The two processing streams are coupled using full -

In [39]:
simp_ans_finder = ScoredAnswerFinder()


question_with_answers3 = simp_ans_finder.find_answers(
        question = convert_into_question_class(questions)[0],
        scored_chunks = scored_3
        )


print("\n---\n---> Founded answers:")
for ans in question_with_answers3.answers:
    print(ans.text)
    print(ans.chunk.score)
    print(ans.confidence)
    print(ans.chunk.chunk.text)
    print("----------------------\n")

Device set to use cuda:1



---
---> Founded answers:
Semantic image segmentation
2.164155
3.913524415111169e-05
Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks. In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference]. Example output and the abstract structure of our fullresolution residual network. The network has two processing streams. The residual stream (blue) stays at the full image resolution, the pooling stream (red) undergoes a sequence of pooling and unpooling operations. The two processing streams are coupled using full - r

In [40]:
simp_ans_finder = ScoredAnswerFinder()


question_with_answers4 = simp_ans_finder.find_answers(
        question = convert_into_question_class(questions)[0],
        scored_chunks = scored_4
        )


print("\n---\n---> Founded answers:")
for ans in question_with_answers4.answers:
    print(ans.text)
    print(ans.chunk.score)
    print(ans.confidence)
    print(ans.chunk.chunk.text)
    print("----------------------\n")

Device set to use cuda:1



---
---> Founded answers:
Semantic image segmentation
1.3244383
3.913524415111169e-05
Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks. In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference]. Example output and the abstract structure of our fullresolution residual network. The network has two processing streams. The residual stream (blue) stays at the full image resolution, the pooling stream (red) undergoes a sequence of pooling and unpooling operations. The two processing streams are coupled using full - 

In [41]:
import numpy as np
from typing import List, Tuple, Dict
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

class AnswerConsensusFinder:
    """
    Класс для нахождения консенсусного ответа на основе оценок фрагментов и сходства между ответами.
    """
    
    def __init__(self):
        """
        Инициализация модели для вычисления эмбеддингов.
        """
        self.model = SentenceTokenizerSingleton()
    
    def find_consensus_answer(self, 
                            answers: List[Answer],
                            chunk_weight: float = 0.6,
                            similarity_weight: float = 0.4,
                            min_similarity_threshold: float = 0.7,
                            top_k_candidates: int = 5) -> Answer:
        """
        Находит консенсусный ответ на основе комбинации оценки фрагмента и сходства с другими ответами.
        
        Args:
            answers: Список объектов Answer
            chunk_weight: Вес оценки фрагмента в финальной оценке (0-1)
            similarity_weight: Вес сходства с другими ответами (0-1)
            min_similarity_threshold: Минимальный порог сходства для учета
            top_k_candidates: Количество топ-кандидатов для анализа
            
        Returns:
            Answer: Наиболее вероятный правильный ответ
        """
        if not answers:
            return None
        
        if len(answers) == 1:
            return answers[0]
        
        # Нормализуем веса
        total_weight = chunk_weight + similarity_weight
        chunk_weight /= total_weight
        similarity_weight /= total_weight
        
        # Шаг 1: Получаем эмбеддинги для всех ответов
        answer_texts = [answer.text for answer in answers]
        embeddings = self.model.tokenizer.encode(answer_texts)
        
        # Шаг 2: Вычисляем матрицу попарных сходств
        similarity_matrix = cosine_similarity(embeddings)
        
        # Шаг 3: Вычисляем оценку консенсуса для каждого ответа
        consensus_scores = []
        
        for i, answer in enumerate(answers):
            # Базовая оценка из фрагмента (нормализованная)
            chunk_score = getattr(answer, 'chunk_score', 0.0)
            confidence_score = getattr(answer, 'confidence', 0.0)
            
            # Комбинированная оценка фрагмента
            base_score = (chunk_score + confidence_score) / 2
            
            # Оценка сходства с другими ответами
            similarity_scores = []
            for j, other_answer in enumerate(answers):
                if i != j:
                    similarity = similarity_matrix[i][j]
                    if similarity >= min_similarity_threshold:
                        # Взвешиваем сходство с важностью другого ответа
                        other_chunk_score = getattr(other_answer, 'chunk_score', 0.0)
                        other_confidence = getattr(other_answer, 'confidence', 0.0)
                        other_importance = (other_chunk_score + other_confidence) / 2
                        
                        weighted_similarity = similarity * other_importance
                        similarity_scores.append(weighted_similarity)
            
            # Средняя оценка сходства (0 если нет похожих ответов)
            avg_similarity = np.mean(similarity_scores) if similarity_scores else 0.0
            
            # Финальная оценка консенсуса
            consensus_score = (base_score * chunk_weight) + (avg_similarity * similarity_weight)
            
            consensus_scores.append(consensus_score)
        
        # Шаг 4: Выбираем ответ с максимальной оценкой консенсуса
        best_index = np.argmax(consensus_scores)
        best_answer = answers[best_index]
        
        # Добавляем информацию о консенсусе
        best_answer.consensus_score = consensus_scores[best_index]
        best_answer.similar_answers_count = sum(1 for i, score in enumerate(consensus_scores) 
                                              if i != best_index and similarity_matrix[best_index][i] >= min_similarity_threshold)
        
        return best_answer

    def find_consensus_with_clustering(self,
                                     answers: List[Answer],
                                     similarity_threshold: float = 0.75) -> Answer:
        """
        Альтернативная стратегия: кластеризация ответов и выбор из наибольшего кластера.
        """
        print(f"=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===")
        print(f"Количество ответов для анализа: {len(answers)}")
        print(f"Порог сходства для кластеризации: {similarity_threshold}")
        
        if not answers:
            print("❌ Пустой список ответов")
            return None
            
        if len(answers) == 1:
            print("✅ Только один ответ - возвращаем его")
            return answers[0]
        
        # Получаем эмбеддинги
        answer_texts = [answer.text for answer in answers]
        print(f"\n📊 Тексты ответов для анализа:")
        for i, text in enumerate(answer_texts):
            chunk_score = getattr(answers[i], 'chunk_score', 0.0)
            confidence = getattr(answers[i], 'confidence', 0.0)
            print(f"  {i+1}. '{text[:50]}{'...' if len(text) > 50 else ''}' "
                  f"(chunk_score: {chunk_score:.3f}, confidence: {confidence:.3f})")
        
        embeddings = self.model.tokenizer.encode(answer_texts)
        print(f"✅ Получены эмбеддинги размерности: {embeddings.shape}")
        
        # Простая кластеризация по сходству (возвращает индексы кластеров)
        cluster_indices = self._cluster_answers(embeddings, similarity_threshold)
        
        print(f"\n🔍 Результаты кластеризации:")
        print(f"Образовано кластеров: {len(cluster_indices)}")
        
        # Преобразуем индексы в кластеры объектов Answer
        clusters = []
        for cluster_idx, indices in enumerate(cluster_indices):
            cluster = [answers[i] for i in indices]
            clusters.append(cluster)
            print(f"Кластер {cluster_idx+1}: {len(indices)} ответов (индексы: {indices})")
            
            # Выводим содержимое кластера
            for i, answer_idx in enumerate(indices):
                answer = answers[answer_idx]
                chunk_score = getattr(answer, 'chunk_score', 0.0)
                confidence = getattr(answer, 'confidence', 0.0)
                print(f"    {i+1}. '{answer.text[:60]}{'...' if len(answer.text) > 60 else ''}' "
                      f"(score: {chunk_score:.3f}, conf: {confidence:.3f})")
        
        # Находим самый большой кластер
        if not clusters:
            print("\n⚠️ Кластеры не образовались - возвращаем ответ с наивысшей оценкой")
            best_answer = max(answers, key=lambda x: getattr(x, 'chunk_score', 0.0))
            chunk_score = getattr(best_answer, 'chunk_score', 0.0)
            print(f"✅ Выбран ответ: '{best_answer.text}' (chunk_score: {chunk_score:.3f})")
            return best_answer
        
        largest_cluster = max(clusters, key=len)
        largest_cluster_size = len(largest_cluster)
        
        print(f"\n📈 Самый большой кластер: {largest_cluster_size} ответов")
        
        if not largest_cluster:
            print("⚠️ Самый большой кластер пуст - возвращаем ответ с наивысшей оценкой")
            best_answer = max(answers, key=lambda x: getattr(x, 'chunk_score', 0.0))
            chunk_score = getattr(best_answer, 'chunk_score', 0.0)
            print(f"✅ Выбран ответ: '{best_answer.text}' (chunk_score: {chunk_score:.3f})")
            return best_answer
        
        # В самом большом кластере выбираем ответ с наивысшей оценкой фрагмента
        print(f"\n🎯 Выбор лучшего ответа в самом большом кластере:")
        best_answer_in_cluster = max(largest_cluster, 
                                   key=lambda x: getattr(x, 'chunk_score', 0.0))
        
        chunk_score = getattr(best_answer_in_cluster, 'chunk_score', 0.0)
        confidence = getattr(best_answer_in_cluster, 'confidence', 0.0)
        
        print(f"✅ Лучший ответ в кластере: '{best_answer_in_cluster.text}'")
        print(f"   - Оценка фрагмента: {chunk_score:.3f}")
        print(f"   - Уверенность модели: {confidence:.3f}")
        print(f"   - Размер кластера: {largest_cluster_size}")
        print(f"   - Всего кластеров: {len(clusters)}")
        
        # Добавляем информацию о кластере
        best_answer_in_cluster.cluster_size = len(largest_cluster)
        best_answer_in_cluster.total_clusters = len(clusters)
        
        print(f"\n✅ ФИНАЛЬНЫЙ ВЫБОР: '{best_answer_in_cluster.text[:80]}{'...' if len(best_answer_in_cluster.text) > 80 else ''}'")
        print("=== ЗАВЕРШЕНИЕ КЛАСТЕРИЗАЦИИ ===")
        
        return best_answer_in_cluster
    
    def _cluster_answers(self, embeddings: np.ndarray, threshold: float) -> List[List[int]]:
        """
        Простая кластеризация ответов на основе косинусного сходства.
        Возвращает список кластеров, где каждый кластер - список индексов ответов.
        """
        n = len(embeddings)
        clusters = []
        visited = set()
        
        similarity_matrix = cosine_similarity(embeddings)
        
        for i in range(n):
            if i in visited:
                continue
                
            cluster = [i]
            visited.add(i)
            
            # Находим все похожие ответы
            for j in range(i + 1, n):
                if j not in visited and similarity_matrix[i][j] >= threshold:
                    cluster.append(j)
                    visited.add(j)
            
            clusters.append(cluster)
        
        return clusters

    # Альтернативная упрощенная версия без кластеризации
    def find_consensus_simple(self, answers: List[Answer]) -> Answer:
        """
        Упрощенная версия: выбирает ответ с наибольшей поддержкой от других похожих ответов.
        """
        if not answers:
            return None
            
        if len(answers) == 1:
            return answers[0]
        
        # Получаем эмбеддинги
        answer_texts = [answer.text for answer in answers]
        embeddings = self.model.tokenizer.encode(answer_texts)
        similarity_matrix = cosine_similarity(embeddings)
        
        # Для каждого ответа вычисляем сумму сходств с другими ответами, взвешенную по их оценкам
        support_scores = []
        
        for i, answer in enumerate(answers):
            chunk_score = getattr(answer, 'chunk_score', 0.0)
            confidence = getattr(answer, 'confidence', 0.0)
            base_score = (chunk_score + confidence) / 2
            
            # Вычисляем поддержку от других ответов
            support = 0.0
            for j, other_answer in enumerate(answers):
                if i != j:
                    other_chunk_score = getattr(other_answer, 'chunk_score', 0.0)
                    other_confidence = getattr(other_answer, 'confidence', 0.0)
                    other_score = (other_chunk_score + other_confidence) / 2
                    
                    support += similarity_matrix[i][j] * other_score
            
            # Комбинированная оценка
            combined_score = 0.7 * base_score + 0.3 * (support / (len(answers) - 1))
            support_scores.append(combined_score)
        
        # Выбираем ответ с максимальной комбинированной оценкой
        best_index = np.argmax(support_scores)
        best_answer = answers[best_index]
        best_answer.consensus_score = support_scores[best_index]
        
        return best_answer

    def extra_find_consensus_with_clustering(self,
                                     answers: List[Answer],
                                     similarity_threshold: float = 0.75,
                                     cluster_selection_strategy: str = "weighted_score",
                                     answer_selection_strategy: str = "highest_chunk_score") -> Answer:
        """
        Альтернативная стратегия: кластеризация ответов и выбор из наилучшего кластера.
        
        Args:
            cluster_selection_strategy: Стратегия выбора кластера:
                - "largest": Самый большой кластер (оригинальная стратегия)
                - "highest_avg_score": Кластер с наивысшим средним chunk_score
                - "weighted_score": Кластер с наивысшим взвешенным score (учитывает размер и качество)
                - "best_single": Кластер, содержащий ответ с наивысшим chunk_score
            answer_selection_strategy: Стратегия выбора ответа внутри кластера:
                - "highest_chunk_score": Ответ с наивысшим chunk_score (по умолчанию)
                - "highest_similarity": Ответ с наивысшим средним сходством с другими ответами в кластере
                - "combined_score": Комбинация chunk_score и сходства
        """
        print(f"=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===")
        print(f"Количество ответов для анализа: {len(answers)}")
        print(f"Порог сходства для кластеризации: {similarity_threshold}")
        print(f"Стратегия выбора кластера: {cluster_selection_strategy}")
        print(f"Стратегия выбора ответа: {answer_selection_strategy}")
        
        if not answers:
            print("❌ Пустой список ответов")
            return None
            
        if len(answers) == 1:
            print("✅ Только один ответ - возвращаем его")
            return answers[0]
        
        # Получаем эмбеддинги
        answer_texts = [answer.text for answer in answers]
        print(f"\n📊 Тексты ответов для анализа:")
        for i, text in enumerate(answer_texts):
            chunk_score = getattr(answers[i], 'chunk_score', 0.0)
            confidence = getattr(answers[i], 'confidence', 0.0)
            print(f"  {i+1}. '{text[:50]}{'...' if len(text) > 50 else ''}' "
                  f"(chunk_score: {chunk_score:.3f}, confidence: {confidence:.3f})")
        
        embeddings = self.model.tokenizer.encode(answer_texts)
        print(f"✅ Получены эмбеддинги размерности: {embeddings.shape}")
        
        # Вычисляем матрицу сходств для всех ответов
        similarity_matrix = cosine_similarity(embeddings)
        print(f"\n🔬 Матрица сходств (первые 5x5):")
        for i in range(min(5, len(answers))):
            row = [f"{similarity_matrix[i][j]:.2f}" for j in range(min(5, len(answers)))]
            print(f"  {i}: {row}")
        
        # Простая кластеризация по сходству (возвращает индексы кластеров)
        cluster_indices = self._cluster_answers(embeddings, similarity_threshold)
        
        print(f"\n🔍 Результаты кластеризации:")
        print(f"Образовано кластеров: {len(cluster_indices)}")
        
        # Преобразуем индексы в кластеры объектов Answer
        clusters = []
        cluster_metrics = []
        
        for cluster_idx, indices in enumerate(cluster_indices):
            cluster = [answers[i] for i in indices]
            clusters.append(cluster)
            
            # Вычисляем метрики для кластера
            chunk_scores = [getattr(answer, 'chunk_score', 0.0) for answer in cluster]
            confidences = [getattr(answer, 'confidence', 0.0) for answer in cluster]
            
            avg_chunk_score = np.mean(chunk_scores) if chunk_scores else 0.0
            avg_confidence = np.mean(confidences) if confidences else 0.0
            max_chunk_score = max(chunk_scores) if chunk_scores else 0.0
            cluster_size = len(cluster)
            
            # Вычисляем среднее сходство внутри кластера
            intra_cluster_similarities = []
            for i, idx_i in enumerate(indices):
                for j, idx_j in enumerate(indices):
                    if i < j:  # Избегаем дублирования и диагонали
                        intra_cluster_similarities.append(similarity_matrix[idx_i][idx_j])
            
            avg_intra_similarity = np.mean(intra_cluster_similarities) if intra_cluster_similarities else 0.0
            
            # Взвешенная оценка кластера (размер * среднее качество)
            weighted_score = cluster_size * avg_chunk_score
            
            cluster_metrics.append({
                'cluster': cluster,
                'size': cluster_size,
                'avg_chunk_score': avg_chunk_score,
                'avg_confidence': avg_confidence,
                'max_chunk_score': max_chunk_score,
                'weighted_score': weighted_score,
                'avg_intra_similarity': avg_intra_similarity,
                'indices': indices,
                'similarity_matrix': similarity_matrix
            })
            
            print(f"\nКластер {cluster_idx+1}:")
            print(f"  Размер: {cluster_size} ответов")
            print(f"  Средний chunk_score: {avg_chunk_score:.3f}")
            print(f"  Средняя уверенность: {avg_confidence:.3f}")
            print(f"  Максимальный chunk_score: {max_chunk_score:.3f}")
            print(f"  Среднее сходство внутри кластера: {avg_intra_similarity:.3f}")
            print(f"  Взвешенная оценка: {weighted_score:.3f}")
            print(f"  Индексы: {indices}")
            
            # Выводим содержимое кластера
            for i, answer in enumerate(cluster):
                chunk_score = getattr(answer, 'chunk_score', 0.0)
                confidence = getattr(answer, 'confidence', 0.0)
                print(f"    {i+1}. '{answer.text[:60]}{'...' if len(answer.text) > 60 else ''}' "
                      f"(score: {chunk_score:.3f}, conf: {confidence:.3f})")
        
        # Находим наилучший кластер по выбранной стратегии
        if not clusters:
            print("\n⚠️ Кластеры не образовались - возвращаем ответ с наивысшей оценкой")
            best_answer = max(answers, key=lambda x: getattr(x, 'chunk_score', 0.0))
            chunk_score = getattr(best_answer, 'chunk_score', 0.0)
            print(f"✅ Выбран ответ: '{best_answer.text}' (chunk_score: {chunk_score:.3f})")
            return best_answer
        
        selected_cluster_metrics = None
        
        if cluster_selection_strategy == "largest":
            selected_cluster_metrics = max(cluster_metrics, key=lambda x: x['size'])
            print(f"\n📈 Стратегия 'largest': выбран самый большой кластер (размер: {selected_cluster_metrics['size']})")
            
        elif cluster_selection_strategy == "highest_avg_score":
            selected_cluster_metrics = max(cluster_metrics, key=lambda x: x['avg_chunk_score'])
            print(f"\n📈 Стратегия 'highest_avg_score': выбран кластер с наивысшим средним score ({selected_cluster_metrics['avg_chunk_score']:.3f})")
            
        elif cluster_selection_strategy == "weighted_score":
            selected_cluster_metrics = max(cluster_metrics, key=lambda x: x['weighted_score'])
            print(f"\n📈 Стратегия 'weighted_score': выбран кластер с наивысшей взвешенной оценкой ({selected_cluster_metrics['weighted_score']:.3f})")
            
        elif cluster_selection_strategy == "best_single":
            global_best_score = max(getattr(answer, 'chunk_score', 0.0) for answer in answers)
            for metrics in cluster_metrics:
                if metrics['max_chunk_score'] == global_best_score:
                    selected_cluster_metrics = metrics
                    break
            if selected_cluster_metrics is None:
                selected_cluster_metrics = max(cluster_metrics, key=lambda x: x['max_chunk_score'])
            print(f"\n📈 Стратегия 'best_single': выбран кластер с наивысшим отдельным ответом (score: {selected_cluster_metrics['max_chunk_score']:.3f})")
        
        elif cluster_selection_strategy == "highest_cohesion":
            # Новая стратегия: кластер с наивысшим внутренним сходством
            selected_cluster_metrics = max(cluster_metrics, key=lambda x: x['avg_intra_similarity'])
            print(f"\n📈 Стратегия 'highest_cohesion': выбран наиболее сплоченный кластер (сходство: {selected_cluster_metrics['avg_intra_similarity']:.3f})")
        
        else:
            selected_cluster_metrics = max(cluster_metrics, key=lambda x: x['weighted_score'])
            print(f"\n📈 Стратегия по умолчанию 'weighted_score': выбран кластер с наивысшей взвешенной оценкой ({selected_cluster_metrics['weighted_score']:.3f})")
        
        selected_cluster = selected_cluster_metrics['cluster']
        cluster_indices_list = selected_cluster_metrics['indices']
        cluster_similarity_matrix = selected_cluster_metrics['similarity_matrix']
        
        # Выбор лучшего ответа в кластере по выбранной стратегии
        print(f"\n🎯 Выбор лучшего ответа в выбранном кластере (стратегия: {answer_selection_strategy}):")
        
        if answer_selection_strategy == "highest_chunk_score":
            # Оригинальная стратегия: наивысший chunk_score
            best_answer_in_cluster = max(selected_cluster, 
                                       key=lambda x: getattr(x, 'chunk_score', 0.0))
            print(f"  Использована стратегия 'highest_chunk_score'")
            
        elif answer_selection_strategy == "highest_similarity":
            # Новая стратегия: ответ с наивысшим средним сходством с другими ответами в кластере
            best_answer_in_cluster = self._select_answer_by_similarity(
                selected_cluster, cluster_indices_list, cluster_similarity_matrix)
            print(f"  Использована стратегия 'highest_similarity'")
            
        elif answer_selection_strategy == "combined_score":
            # Комбинированная стратегия: chunk_score и сходство
            best_answer_in_cluster = self._select_answer_by_combined_score(
                selected_cluster, cluster_indices_list, cluster_similarity_matrix)
            print(f"  Использована стратегия 'combined_score'")
            
        else:
            best_answer_in_cluster = max(selected_cluster, 
                                       key=lambda x: getattr(x, 'chunk_score', 0.0))
            print(f"  Использована стратегия по умолчанию 'highest_chunk_score'")
        
        chunk_score = getattr(best_answer_in_cluster, 'chunk_score', 0.0)
        confidence = getattr(best_answer_in_cluster, 'confidence', 0.0)
        
        print(f"✅ Лучший ответ в кластере: '{best_answer_in_cluster.text}'")
        print(f"   - Оценка фрагмента: {chunk_score:.3f}")
        print(f"   - Уверенность модели: {confidence:.3f}")
        print(f"   - Размер кластера: {selected_cluster_metrics['size']}")
        print(f"   - Средний chunk_score кластера: {selected_cluster_metrics['avg_chunk_score']:.3f}")
        print(f"   - Среднее сходство в кластере: {selected_cluster_metrics['avg_intra_similarity']:.3f}")
        print(f"   - Всего кластеров: {len(clusters)}")
        print(f"   - Стратегия выбора кластера: {cluster_selection_strategy}")
        print(f"   - Стратегия выбора ответа: {answer_selection_strategy}")
        
        # Добавляем информацию о кластере
        best_answer_in_cluster.cluster_size = selected_cluster_metrics['size']
        best_answer_in_cluster.total_clusters = len(clusters)
        best_answer_in_cluster.cluster_avg_score = selected_cluster_metrics['avg_chunk_score']
        best_answer_in_cluster.cluster_avg_similarity = selected_cluster_metrics['avg_intra_similarity']
        best_answer_in_cluster.selection_strategy = f"{cluster_selection_strategy}+{answer_selection_strategy}"
        
        print(f"\n✅ ФИНАЛЬНЫЙ ВЫБОР: '{best_answer_in_cluster.text[:80]}{'...' if len(best_answer_in_cluster.text) > 80 else ''}'")
        print("=== ЗАВЕРШЕНИЕ КЛАСТЕРИЗАЦИИ ===")
        
        return best_answer_in_cluster

    def _select_answer_by_similarity(self, cluster: List[Answer], indices: List[int], 
                                   similarity_matrix: np.ndarray) -> Answer:
        """
        Выбирает ответ с наивысшим средним сходством с другими ответами в кластере.
        """
        print(f"  Поиск ответа с максимальным средним сходством в кластере:")
        
        best_answer = None
        best_avg_similarity = -1
        
        for i, answer in enumerate(cluster):
            original_index = indices[i]
            similarities = []
            
            for j, other_answer in enumerate(cluster):
                if i != j:
                    other_original_index = indices[j]
                    similarity = similarity_matrix[original_index][other_original_index]
                    similarities.append(similarity)
            
            avg_similarity = np.mean(similarities) if similarities else 0.0
            chunk_score = getattr(answer, 'chunk_score', 0.0)
            
            print(f"    Ответ {i+1}: '{answer.text[:40]}...' - среднее сходство: {avg_similarity:.3f}, chunk_score: {chunk_score:.3f}")
            
            if avg_similarity > best_avg_similarity:
                best_avg_similarity = avg_similarity
                best_answer = answer
        
        print(f"  🎯 Выбран ответ со средним сходством: {best_avg_similarity:.3f}")
        return best_answer

    def _select_answer_by_combined_score(self, cluster: List[Answer], indices: List[int],
                                       similarity_matrix: np.ndarray) -> Answer:
        """
        Выбирает ответ на основе комбинации chunk_score и сходства.
        """
        print(f"  Поиск ответа по комбинированной оценке:")
        
        best_answer = None
        best_combined_score = -1
        
        for i, answer in enumerate(cluster):
            original_index = indices[i]
            
            # Вычисляем среднее сходство
            similarities = []
            for j, other_answer in enumerate(cluster):
                if i != j:
                    other_original_index = indices[j]
                    similarity = similarity_matrix[original_index][other_original_index]
                    similarities.append(similarity)
            
            avg_similarity = np.mean(similarities) if similarities else 0.0
            chunk_score = getattr(answer, 'chunk_score', 0.0)
            
            # Комбинированная оценка (50% chunk_score + 50% сходство)
            combined_score = 0.5 * chunk_score + 0.5 * avg_similarity
            
            print(f"    Ответ {i+1}: '{answer.text[:40]}...' - combined: {combined_score:.3f} "
                  f"(chunk: {chunk_score:.3f}, similarity: {avg_similarity:.3f})")
            
            if combined_score > best_combined_score:
                best_combined_score = combined_score
                best_answer = answer
        
        print(f"  🎯 Выбран ответ с комбинированной оценкой: {best_combined_score:.3f}")
        return best_answer

In [42]:
ans_con_finder = AnswerConsensusFinder()

In [43]:
con_ans1 = ans_con_finder.find_consensus_answer( 
                            answers = question_with_answers1.answers)

con_ans1

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str

In [44]:
con_ans2 = ans_con_finder.find_consensus_with_clustering( 
                            answers = question_with_answers1.answers)

con_ans2

=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===
Количество ответов для анализа: 16
Порог сходства для кластеризации: 0.75

📊 Тексты ответов для анализа:
  1. 'Semantic image segmentation' (chunk_score: 4.263, confidence: 0.000)
  2. 'semantic segmentation' (chunk_score: 1.799, confidence: 0.001)
  3. 'segmentation' (chunk_score: 1.243, confidence: 0.000)
  4. 'assigning a set of predefined class labels to imag...' (chunk_score: 1.190, confidence: 0.000)
  5. 'computes a function F' (chunk_score: 0.882, confidence: 0.000)
  6. 'By using the two processing streams together' (chunk_score: 0.657, confidence: 0.000)
  7. 'stereo or optical flow' (chunk_score: 0.331, confidence: 0.011)
  8. 'state - of - the - art results even without pre - ...' (chunk_score: 0.292, confidence: 0.000)
  9. 'smoothing the network predictions' (chunk_score: 0.289, confidence: 0.001)
  10. 'bootstrapped cross - entropy loss' (chunk_score: 0.269, confidence: 0.003)
  11. 'FRRN B' (chunk_score: 0.223, confidence: 0.000)
  1

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str

In [45]:
con_ans3 = ans_con_finder.find_consensus_simple( 
                            answers = question_with_answers1.answers)

con_ans3

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str

In [46]:
# Сбалансированный подход (рекомендуется)
con_ans4 = ans_con_finder.extra_find_consensus_with_clustering(
    answers=question_with_answers1.answers,
    similarity_threshold=0.7,
    cluster_selection_strategy="weighted_score"
)
con_ans4

=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===
Количество ответов для анализа: 16
Порог сходства для кластеризации: 0.7
Стратегия выбора кластера: weighted_score
Стратегия выбора ответа: highest_chunk_score

📊 Тексты ответов для анализа:
  1. 'Semantic image segmentation' (chunk_score: 4.263, confidence: 0.000)
  2. 'semantic segmentation' (chunk_score: 1.799, confidence: 0.001)
  3. 'segmentation' (chunk_score: 1.243, confidence: 0.000)
  4. 'assigning a set of predefined class labels to imag...' (chunk_score: 1.190, confidence: 0.000)
  5. 'computes a function F' (chunk_score: 0.882, confidence: 0.000)
  6. 'By using the two processing streams together' (chunk_score: 0.657, confidence: 0.000)
  7. 'stereo or optical flow' (chunk_score: 0.331, confidence: 0.011)
  8. 'state - of - the - art results even without pre - ...' (chunk_score: 0.292, confidence: 0.000)
  9. 'smoothing the network predictions' (chunk_score: 0.289, confidence: 0.001)
  10. 'bootstrapped cross - entropy loss' (chunk_score

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str

In [47]:
# Акцент на качество
con_ans5 = ans_con_finder.extra_find_consensus_with_clustering(
    answers=question_with_answers1.answers,
    cluster_selection_strategy="highest_avg_score"
)
con_ans5

=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===
Количество ответов для анализа: 16
Порог сходства для кластеризации: 0.75
Стратегия выбора кластера: highest_avg_score
Стратегия выбора ответа: highest_chunk_score

📊 Тексты ответов для анализа:
  1. 'Semantic image segmentation' (chunk_score: 4.263, confidence: 0.000)
  2. 'semantic segmentation' (chunk_score: 1.799, confidence: 0.001)
  3. 'segmentation' (chunk_score: 1.243, confidence: 0.000)
  4. 'assigning a set of predefined class labels to imag...' (chunk_score: 1.190, confidence: 0.000)
  5. 'computes a function F' (chunk_score: 0.882, confidence: 0.000)
  6. 'By using the two processing streams together' (chunk_score: 0.657, confidence: 0.000)
  7. 'stereo or optical flow' (chunk_score: 0.331, confidence: 0.011)
  8. 'state - of - the - art results even without pre - ...' (chunk_score: 0.292, confidence: 0.000)
  9. 'smoothing the network predictions' (chunk_score: 0.289, confidence: 0.001)
  10. 'bootstrapped cross - entropy loss' (chunk_s

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str

In [48]:
# Ответ, который наиболее репрезентативен для кластера
con_ans6 = ans_con_finder.extra_find_consensus_with_clustering(
    answers=question_with_answers1.answers,
    cluster_selection_strategy="weighted_score",
    answer_selection_strategy="highest_similarity"
)
con_ans6

=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===
Количество ответов для анализа: 16
Порог сходства для кластеризации: 0.75
Стратегия выбора кластера: weighted_score
Стратегия выбора ответа: highest_similarity

📊 Тексты ответов для анализа:
  1. 'Semantic image segmentation' (chunk_score: 4.263, confidence: 0.000)
  2. 'semantic segmentation' (chunk_score: 1.799, confidence: 0.001)
  3. 'segmentation' (chunk_score: 1.243, confidence: 0.000)
  4. 'assigning a set of predefined class labels to imag...' (chunk_score: 1.190, confidence: 0.000)
  5. 'computes a function F' (chunk_score: 0.882, confidence: 0.000)
  6. 'By using the two processing streams together' (chunk_score: 0.657, confidence: 0.000)
  7. 'stereo or optical flow' (chunk_score: 0.331, confidence: 0.011)
  8. 'state - of - the - art results even without pre - ...' (chunk_score: 0.292, confidence: 0.000)
  9. 'smoothing the network predictions' (chunk_score: 0.289, confidence: 0.001)
  10. 'bootstrapped cross - entropy loss' (chunk_score

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str

In [49]:
# Сбалансированный подход
con_ans7 = ans_con_finder.extra_find_consensus_with_clustering(
    answers=question_with_answers1.answers,
    cluster_selection_strategy="highest_cohesion",
    answer_selection_strategy="combined_score"
)
con_ans7

=== НАЧАЛО КЛАСТЕРИЗАЦИИ ОТВЕТОВ ===
Количество ответов для анализа: 16
Порог сходства для кластеризации: 0.75
Стратегия выбора кластера: highest_cohesion
Стратегия выбора ответа: combined_score

📊 Тексты ответов для анализа:
  1. 'Semantic image segmentation' (chunk_score: 4.263, confidence: 0.000)
  2. 'semantic segmentation' (chunk_score: 1.799, confidence: 0.001)
  3. 'segmentation' (chunk_score: 1.243, confidence: 0.000)
  4. 'assigning a set of predefined class labels to imag...' (chunk_score: 1.190, confidence: 0.000)
  5. 'computes a function F' (chunk_score: 0.882, confidence: 0.000)
  6. 'By using the two processing streams together' (chunk_score: 0.657, confidence: 0.000)
  7. 'stereo or optical flow' (chunk_score: 0.331, confidence: 0.011)
  8. 'state - of - the - art results even without pre - ...' (chunk_score: 0.292, confidence: 0.000)
  9. 'smoothing the network predictions' (chunk_score: 0.289, confidence: 0.001)
  10. 'bootstrapped cross - entropy loss' (chunk_score: 

Answer(text='Semantic image segmentation', chunk=ScoredChunk(chunk=TextChunk(sentences=[Sentence(text='Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks.', num=13), Sentence(text='In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference].', num=14), Sentence(text='Example output and the abstract structure of our fullresolution residual network.', num=15), Sentence(text='The network has two processing streams.', num=16), Sentence(text='The residual stream (blue) stays at the full image resolution, the pooling str